Conda env uses conda-forge: python 3.15.2, pandas, geopandas, requests

References:
- https://www.eastbaytimes.com/2004/05/24/whats-in-a-street-name-new-guidebook-finds-out/
- https://geopandas.org/en/stable/getting_started/introduction.html#Making-maps 
- https://andreagordon.com/blog/the-secret-history-behind-berkeleys-street-names 
- https://berkeleyca.gov/city-services/community-gis-portal
- https://city-of-berkeley.hub.arcgis.com/ 
- https://gis.cityofberkeley.info/arcgis3/rest/services/Geocode/ 

The goal of this notebook is to review all location-dependent data for all datasets. This will allow me to map out necessary assumptions about the inputs and output for converting all location data to coordinate values.

In [ ]:
import pandas as pd
from IPython.display import display

In [ ]:
# loading in all data sets
PATH_TO_DATA = "../data/raw/"

files = {
    "cases": "arrests_2016_current.csv",
    "force": "use_of_force_2021_current.csv",
    "stops_1": "stop_data_2020_2023.csv",
    "stops_2": "stop_data_2024_current.csv",
}

data = {name : pd.read_csv(PATH_TO_DATA + filename) for name, filename in files.items()}

In [ ]:
pd.DataFrame([{"dataset" : name, 
               "rows" : len(frame),
               "columns" : len(frame.columns)}
               for name, frame in data.items()])

In [ ]:
for name, frame in data.items():
    print(name)
    print(frame.columns.tolist())
    display(frame.head(5))

# there are so many different address names.... like
# cases: `Block_Address`
# force: `Block Address`
# stops_1: `LocationDescription`
# stops_2: a bunch of separate location fields...
# 
# Also I see `NonBerkeley Address` in force thats binary, no location specified. 
# and the force data shows a block wording repeated in kains street: 1200 Block of block of Kains

In [ ]:
# only looking into the location data for each dataset now 
location_data = {
    "cases" : ["Block_Address", "City", "ZIP_Code"],
    "force" : ["Block Address", "City", "NonBerkeley Address"],
    "stops_1" : ["Latitude", "Longitude", "LocationDescription", "City"],
    "stops_2" : ["LocationType", "Latitude", "Longitude", "BlockNumber", "StreetName", "CrossStreet1", "CrossStreet2", 
                 "Highway", "ClosestExit", "OtherLocationDesc", "City"]
}

# checking if there are some missing elements or columsn that may not be worth using as inputs
for name, columns in location_data.items():
    values = data[name][columns]
    print(name)
    display(pd.DataFrame({"missing" : values.isna().sum(), 
                         "unique counts" : values.nunique()}))
    display(values.sample(n=min(4, len(values)), random_state=64))

In [ ]:
# looks like stop data has coordinate data already. should check if these are consistent

for name in ["stops_1", "stops_2"]:
    frame = data[name]
    lat = pd.to_numeric(frame["Latitude"], errors="coerce")
    lon = pd.to_numeric(frame["Longitude"], errors="coerce")
    valid = (lat.between(-90, 90) & lon.between(-180, 180) & ~(lat.eq(0) & lon.eq(0)).fillna(False))

    print(name, {
        "valid_pair" : int(valid.sum()),
        "missing_invalid" : int((~valid).sum()) 
    })
    display(frame.loc[~valid, location_data[name]].head(15))

    # stops_1 has 14 bad pairs, stops_2 has 87 bad pairs... locationtype helps in stops_2 but not stops_1. 
    # stops_1 has location description for the cross street or the block number, ok to do case by case?
    # stops_2 have associated locationtype to inform the rule for obtaining the coordinates

In [ ]:
# checking the text format for parsing rules
for name, column in [
    ("cases", "Block_Address"),
    ("force", "Block Address"),
    ("stops_1", "LocationDescription"),
]:
    print(name)
    display(data[name][column].value_counts(dropna=False).head(15))

display(data["stops_2"]["LocationType"].value_counts(dropna=False))

for name, frame in data.items():
    print(name)
    display(frame["City"].str.strip().str.upper().value_counts(dropna=False))

# many different interpretations found like:
# 1. approx block location
# 2. cross street intersection
# 3. numbered address
# 4. landmarks (need separate handling)
# 5. i see a unique situation where the text is obscured or something: `ASH00 AVE & WEST FRONTAGE RD`

# Assumptions about the data
- Addresses containing numbers with `00` at the end of the number should be treated as block numbers. Ex. `"2100 SHATTUCK AVE"` will be treated as `"2100 BLOCK OF SHATTUCK AVE"`. 
- Separators between streets are to be treated as intersections.
- Coordinates that pass the range check will be treated as an adequate representation of the event location.
- The coordinates supplied by the Stop data are treated as World Geodetic System 1984 (WGS84) coordinates, using angular units where latitude and longitude in degrees. This will also be the chosen output convention (note that it is treated as interchangable with type EPSG registry code for the 2D geographic coordinate reference system (EPSG:4326)). 
- `(0,0)` is treated as a missing location in the datasets since it can't represent a Berkeley location.
- The `City` column supplies appropriate geocoding context, therefore the row is assumed to be within Berkeley's boundary. 

# Parsing rules for preprocessing 
- Missing values, empty strings, and whitespace strings treated as unavailable location components
- Standardizing strings by normalizing capitalization and removing extra whitespace
- Make all `BlockNumber` values integer. Zero is a valid block number
- Descriptions for locations involving blocks are converted to block number first then `BLOCK OF [STREET NAME]`. Repeated wording for block was found in an element for force data. Ex. `"1200 Block of block of Kains"` becomes `"1200 BLOCK OF KAINS"`. 
- Intersections names are characterized by their separators `/`, `&`, `and`. Ex. `"Allston Way/Milvia Street"` becomes `"ALLSTON WAY & MILVIA STREET"`
- Numbered addresses without block wording will be kept the same, just standardized by address number and all caps. 
- The newer Stop data, with `LocationType`, will determine the rules which describe the location. Specifically:
  - `LatLonPt` --> `Latitude` and `Longitude`.
  - `BlkNumber` --> `BlockNumber` and `StreetName`.
  - `Intersection` --> `CrossStreet1` and `CrossStreet2`.
  - `HighwayAndExit` --> highway/exit description requiring separate handling.
  - `Other` --> classify `OtherLocationDesc` using the text rules.
- Looking at location descriptions, if it contains the words `near`, `between`, `ifo` (I suspect meaning "in front of..."), the location will not be treated as an exact location description. They will remain unresolved by the initial automatic parsing since they describe proximity, not one point.
- Obscured or uncertain naming will be removed from analysis since we cannot reconstruct or replace them without obtaining more information.  

# Rules for converting to the output coordinates for all locations
-